####  graphlib 格式的数据图：
            t [顶点数] [边数]
            v [顶点id] [顶点标签] [顶点度数]
            e [顶点id1] [顶点id2]
            t 195812 377658
            v 0 0 1031
            v 1 0 1093
            v 2 0 679
            e 15192 43896
            e 15192 53805
上面这个图中顶点有标签（0，1，2），边没有标签，我想自定义扩展标签，现将不同标签放到不同桶里，得到三个桶，在三个桶内为每个顶点分配新的标签，三个桶内设置不同参数控制桶内不同标签的数量，最后讲这些新生成的标签从0编号，例如先按照(0,1,2)划分成了三个桶(0)，(1)，(2)，在这三个桶内参数设置为2，2，3，因此为三个桶内的顶点，重新分配2，2，3种标签，即(0,1),(0,1),(0,1,2),最后合并为（0，1，2，3，4，5，6），你懂我意思吗?这样还是防止新编的标签覆盖原有的标签，将新生成的图命名为heterization_data.graph

#### 1.1 在生成查询的同时要保证用户指定的某一个标签l的节点在查询图中出现少于k次（可以出现1，2，3，4，5。。。k,但k越大生成的概率越小）

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
generate_queries_controlled.py

从带节点标签的图自动生成查询 workload。
功能：
 - 支持生成 star, path, cycle, tree 四种结构。
 - 可配置查询的节点数和每种配置的生成数量。
 - 可选：约束某个指定标签(controlled_label)的节点出现次数，
   出现次数越多，被接受的概率越低。
"""

import os
import random
from collections import deque
from typing import List, Optional, Tuple
import networkx as nx
import math

def load_graph_fastest(filepath: str) -> nx.Graph:
    """
    读取 Fastest .graph（或类似）格式的图。
    期望 v 行： v <vid> <label> [deg]
            e 行： e <u> <v> [label?]
    只读取节点标签和无向边。
    """
    G = nx.Graph()
    if not os.path.exists(filepath):
        raise FileNotFoundError(filepath)
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if parts[0] == "v" and len(parts) >= 3:
                try:
                    vid = int(parts[1])
                    label = int(parts[2])
                    G.add_node(vid, label=label)
                except ValueError:
                    print(f"[警告] 无法解析节点行: {line}")
            elif parts[0] == "e" and len(parts) >= 3:
                try:
                    u = int(parts[1])
                    v = int(parts[2])
                    G.add_edge(u, v)
                except ValueError:
                    print(f"[警告] 无法解析边行: {line}")
    return G


class QueryGenerator:
    """
    一个用于从大型数据图中生成受约束的查询子图的类。
    """
    def __init__(
        self, 
        G: nx.Graph, 
        out_dir: str = "query_graphs", 
        seed: int = 42,
        controlled_label: Optional[int] = None,
        max_k: int = 5,
        decay_factor: float = 0.5
    ):
        """
        初始化查询生成器。

        参数:
            G (nx.Graph): 输入的数据图，节点应包含 'label' 属性。
            out_dir (str): 保存生成查询的输出目录。
            seed (int): 随机数种子。
            controlled_label (Optional[int]): 用户指定的、需要控制出现次数的标签。如果为None，则不进行特殊控制。
            max_k (int): controlled_label 出现的最大允许次数。
            decay_factor (float): 指数衰减的基数，(0, 1)之间，越小衰减越快。
        """
        self.G = G
        self.out_dir = out_dir
        os.makedirs(self.out_dir, exist_ok=True)
        
        # 按标签对节点进行分组，以加速某些操作
        self.nodes_by_label = {}
        for n, d in self.G.nodes(data=True):
            lab = d.get("label", None)
            if lab is not None:
                self.nodes_by_label.setdefault(lab, []).append(n)
        
        random.seed(seed)
        
        self.controlled_label = controlled_label
        self.max_k = max_k
        self.decay_factor = decay_factor

        if controlled_label is not None:
            print(f"[配置] 控制标签: {controlled_label}, 最大出现次数: {max_k}, 衰减因子: {decay_factor}")
            if controlled_label not in self.nodes_by_label:
                print(f"[警告] 数据图中不存在受控标签 {controlled_label}。")


    def _write_fastest_query(self, qnodes: List[int], edges: List[Tuple[int, int]], outpath: str):
        """
        将查询图写成 Fastest .graph 格式。
        节点会重新编号为 0..k-1。
        """
        node_index = {n: i for i, n in enumerate(qnodes)}
        
        # 计算节点在查询子图中的度数
        deg = {n: 0 for n in qnodes}
        for (u, v) in edges:
            if u in deg and v in deg:
                deg[u] += 1
                deg[v] += 1

        with open(outpath, "w") as fout:
            fout.write(f"t {len(qnodes)} {len(edges)}\n")
            for n in qnodes:
                i = node_index[n]
                lab = int(self.G.nodes[n].get("label", 0))
                d = deg[n]
                fout.write(f"v {i} {lab} {d}\n")
            for (u, v) in edges:
                if u in node_index and v in node_index:
                    fout.write(f"e {node_index[u]} {node_index[v]}\n")

    def _induced_subgraph_edges(self, nodes: List[int]) -> List[Tuple[int, int]]:
        """返回给定节点集的诱导子图的边列表。"""
        subgraph = self.G.subgraph(nodes)
        return list(subgraph.edges())

    def _count_controlled_label(self, nodes: List[int]) -> int:
        """计算节点集合中受控标签的数量。"""
        if self.controlled_label is None:
            return 0
        return sum(1 for n in nodes if int(self.G.nodes[n].get("label", 0)) == self.controlled_label)

    def _probabilistic_check_controlled_label(self, nodes: List[int]) -> bool:
        """
        带概率地检查一个节点集合是否被接受。
        这是实现带概率约束的核心函数。
        """
        if self.controlled_label is None:
            return True

        count = self._count_controlled_label(nodes)

        if count == 0 or count > self.max_k:
            return False
            
        # 概率模型: P(accept|count) = decay_factor^(count - 1)
        acceptance_prob = math.pow(self.decay_factor, count - 1)
        
        return random.random() < acceptance_prob

    # ------------------- 结构生成器 -------------------

    def gen_star_one(self, size: int, max_attempts: int = 500) -> Optional[List[int]]:
        """尝试生成一个星形的连通节点集。"""
        for _ in range(max_attempts):
            center = random.choice(list(self.G.nodes()))
            neighbors = list(self.G.neighbors(center))
            
            nodes = [center]
            if len(neighbors) >= size - 1:
                nodes.extend(random.sample(neighbors, size - 1))
            else:
                # 邻居不够时，从2跳邻居中补充
                nodes.extend(neighbors)
                two_hop_neighbors = set()
                for n in neighbors:
                    for n2 in self.G.neighbors(n):
                        if n2 not in nodes:
                            two_hop_neighbors.add(n2)
                
                needed = size - len(nodes)
                if len(two_hop_neighbors) >= needed:
                    nodes.extend(random.sample(list(two_hop_neighbors), needed))
            
            if len(nodes) == size and nx.is_connected(self.G.subgraph(nodes)) and self._probabilistic_check_controlled_label(nodes):
                return nodes
        return None

    def gen_path_one(self, size: int, max_attempts: int = 500) -> Optional[List[int]]:
        """通过随机游走生成一个简单路径。"""
        for _ in range(max_attempts):
            start_node = random.choice(list(self.G.nodes()))
            path = [start_node]
            visited = {start_node}
            
            while len(path) < size:
                current_node = path[-1]
                potential_next = [n for n in self.G.neighbors(current_node) if n not in visited]
                if not potential_next:
                    break
                next_node = random.choice(potential_next)
                path.append(next_node)
                visited.add(next_node)
            
            if len(path) == size and self._probabilistic_check_controlled_label(path):
                return path
        return None

    def gen_cycle_one(self, size: int, max_attempts: int = 800) -> Optional[List[int]]:
        """
        尝试通过随机游走并寻找闭环来生成一个环状子图。
        如果找不到完美的环，则退化为寻找一个稠密的连通子图。
        """
        for _ in range(max_attempts):
            start_node = random.choice(list(self.G.nodes()))
            
            # --- 尝试通过随机游走寻找一个简单环 ---
            path = [start_node]
            visited = {start_node}
            
            while len(path) < size:
                current_node = path[-1]
                # 优先选择未访问过的邻居
                potential_next = [n for n in self.G.neighbors(current_node) if n not in visited]
                
                if not potential_next:
                    # 如果没有未访问的邻居，检查是否可以与路径中较早的非相邻节点形成环
                    # (这是一个复杂的逻辑，我们简化处理，直接中断本次游走)
                    break 
                
                next_node = random.choice(potential_next)
                path.append(next_node)
                visited.add(next_node)

            # 检查游走是否成功生成了足够长的路径，并且首尾相连
            if len(path) == size and self.G.has_edge(path[-1], path[0]):
                if self._probabilistic_check_controlled_label(path):
                    return path

            # --- 如果找不到完美环，退化为寻找一个稠密的连通子图 ---
            # (这个逻辑可以作为备选方案)
            # 我们可以复用 gen_tree_one 来快速获取一个连通节点集
            if len(path) != size: # 如果上面的游走失败了，就重新生成一个节点集
                candidate_nodes = self.gen_tree_one(size, max_attempts=1)
            else: # 如果游走成功但未闭环，就用这个路径上的节点
                candidate_nodes = path

            if candidate_nodes and len(candidate_nodes) == size:
                 subgraph = self.G.subgraph(candidate_nodes)
                 # 稠密条件：边的数量至少等于节点的数量
                 if len(subgraph.edges()) >= size:
                     if self._probabilistic_check_controlled_label(candidate_nodes):
                         return candidate_nodes
                         
        return None

    def gen_tree_one(self, size: int, max_attempts: int = 800) -> Optional[List[int]]:
        """通过广度优先扩展生成一个树状连通子图。"""
        for _ in range(max_attempts):
            start_node = random.choice(list(self.G.nodes()))
            tree_nodes = {start_node}
            frontier = deque([start_node])
            
            while len(tree_nodes) < size and frontier:
                current_node = frontier.popleft()
                neighbors = [n for n in self.G.neighbors(current_node) if n not in tree_nodes]
                random.shuffle(neighbors)
                
                # 每次扩展的节点数
                expand_count = random.randint(1, 3)
                for neighbor in neighbors[:expand_count]:
                    if len(tree_nodes) < size:
                        tree_nodes.add(neighbor)
                        frontier.append(neighbor)
                    else:
                        break
            
            if len(tree_nodes) == size and self._probabilistic_check_controlled_label(list(tree_nodes)):
                return list(tree_nodes)
        return None

    def generate_all(self, sizes: List[int], structures: List[str], N_per: int = 5, max_total_attempts_per_item: int = 1000):
        """
        批量生成所有指定配置的查询图。
        """
        generator_map = {
            "star": self.gen_star_one,
            "path": self.gen_path_one,
            "cycle": self.gen_cycle_one,
            "tree": self.gen_tree_one,
        }

        for struct in structures:
            if struct not in generator_map:
                print(f"[警告] 未知的结构 '{struct}', 已跳过。")
                continue
            gen_func = generator_map[struct]
            for size in sizes:
                out_subdir = os.path.join(self.out_dir, f"{struct}_{size}")
                os.makedirs(out_subdir, exist_ok=True)
                produced = 0
                tries = 0
                seen_node_sets = set()
                print(f"\n[INFO] 正在为 structure={struct}, size={size} 生成 {N_per} 个查询...")
                while produced < N_per and tries < max_total_attempts_per_item:
                    tries += 1
                    nodes = gen_func(size)
                    
                    if not nodes or len(nodes) != size:
                        continue
                        
                    nodes_sorted_tuple = tuple(sorted(nodes))
                    if nodes_sorted_tuple in seen_node_sets:
                        continue
                    
                    seen_node_sets.add(nodes_sorted_tuple)
                    
                    edges = self._induced_subgraph_edges(nodes)
                    fname = os.path.join(out_subdir, f"query_{struct}_{size}_{produced}.graph")
                    self._write_fastest_query(nodes, edges, fname)
                    produced += 1
                
                if produced < N_per:
                    print(f"[警告] 在 {tries} 次尝试后，只为 {struct} size {size} 生成了 {produced}/{N_per} 个查询。")
                else:
                    print(f"[完成] 为 {struct} size {size} 生成了 {produced}/{N_per} 个查询 (尝试次数={tries})。")

# ------------------- 运行入口 -------------------
if __name__ == "__main__":
    # --- 1. 配置区 ---
    DATASET_NAME = 'dataset_test'
    DATA_GRAPH_PATH = f"/home/wangshuo/resource/datasets/parler_data/{DATASET_NAME}/data_graph/parler.graph"
    OUTPUT_DIR = "/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/controlled_generation_v2"
    
    # --- 生成参数 ---
    QUERY_SIZES = [4, 6, 8]
    STRUCTURES = ["star", "path", "cycle", "tree"]
    NUM_QUERIES_PER_CONFIG = 50 # 每种配置生成100个查询
    
    # --- 概率约束配置 ---
    # 如果不希望有任何约束，将 CONTROLLED_LABEL 设置为 None
    CONTROLLED_LABEL = 2        # 您想控制的标签，例如标签 2
    MAX_K = 3                 # 标签2最多出现5次
    DECAY_FACTOR = 0.1         # 概率衰减因子。P = 0.6^(count-1)
    RANDOM_SEED = 2025

    # --- 2. 执行 ---
    print(f"[INFO] 正在加载数据图: {DATA_GRAPH_PATH} ...")
    try:
        data_graph = load_graph_fastest(DATA_GRAPH_PATH)
        print(f"图加载完成: {data_graph.number_of_nodes()} 个节点, {data_graph.number_of_edges()} 条边。")

        generator = QueryGenerator(
            G=data_graph, 
            out_dir=OUTPUT_DIR, 
            seed=RANDOM_SEED,
            controlled_label=CONTROLLED_LABEL,
            max_k=MAX_K,
            decay_factor=DECAY_FACTOR
        )
        
        generator.generate_all(
            sizes=QUERY_SIZES, 
            structures=STRUCTURES, 
            N_per=NUM_QUERIES_PER_CONFIG,
            max_total_attempts_per_item=000 # 可以增加总尝试次数
        )
        
        print("\n[全部完成] 所有查询已生成到目录:", OUTPUT_DIR)

        # --- 3. (可选) 验证生成结果的分布 ---
        print("\n--- 验证生成查询中受控标签的分布 ---")
        from collections import Counter
        label_counts = Counter()
        total_queries = 0
        for root, _, files in os.walk(OUTPUT_DIR):
            for f in files:
                if f.endswith('.graph'):
                    total_queries += 1
                    q_graph = load_graph_fastest(os.path.join(root, f))
                    count = sum(1 for _, d in q_graph.nodes(data=True) if d.get('label') == CONTROLLED_LABEL)
                    label_counts[count] += 1
        
        if total_queries > 0:
            print(f"在总共 {total_queries} 个生成的查询中，标签 {CONTROLLED_LABEL} 的数量分布如下:")
            for count, num_queries in sorted(label_counts.items()):
                percentage = (num_queries / total_queries) * 100
                print(f"  数量为 {count} 的查询有: {num_queries} 个 ({percentage:.2f}%)")
        else:
            print("未找到任何生成的查询文件进行验证。")
            
    except FileNotFoundError as e:
        print(f"[致命错误] 输入文件未找到: {e}")
    except Exception as e:
        print(f"[致命错误] 发生未知异常: {e}")

### 1.1.2  指定生成label=1 的节点只出现一次查询（可以自由设置label），查询节点数从4, 6, 8，10，每种配置生成50个

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
generate_queries_single_label.py

功能：
 - 指定某个特定标签（如 label=1）在查询图中只出现 N 次（如 1 次）。
 - 查询节点数：4, 6, 8, 10。
 - 结构：star, path, cycle, tree。
 - 数量：每种配置 50 个。
"""

import os
import random
from collections import deque, Counter
from typing import List, Optional, Tuple, Dict
import networkx as nx

def load_graph_fastest(filepath: str) -> nx.Graph:
    """
    读取 Fastest .graph 格式。
    """
    G = nx.Graph()
    if not os.path.exists(filepath):
        raise FileNotFoundError(filepath)
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if parts[0] == "v" and len(parts) >= 3:
                try:
                    vid = int(parts[1])
                    label = int(parts[2])
                    G.add_node(vid, label=label)
                except ValueError:
                    pass
            elif parts[0] == "e" and len(parts) >= 3:
                try:
                    u = int(parts[1])
                    v = int(parts[2])
                    G.add_edge(u, v)
                except ValueError:
                    pass
    return G


class QueryGenerator:
    def __init__(
        self, 
        G: nx.Graph, 
        out_dir: str = "query_graphs", 
        seed: int = 42,
        target_constraints: Optional[Dict[int, int]] = None
    ):
        """
        参数:
            target_constraints: 字典 {label_id: count}
            例如 {1: 1} 表示 Label 1 必须出现且仅出现 1 次。
        """
        self.G = G
        self.out_dir = out_dir
        os.makedirs(self.out_dir, exist_ok=True)
        random.seed(seed)
        
        self.target_constraints = target_constraints if target_constraints else {}
        
        # --- 策略优化：预先索引目标节点 ---
        # 我们只关心约束中涉及的那个标签，把它们作为种子节点
        self.candidate_seeds = []
        
        if self.target_constraints:
            print(f"[配置] 严格约束模式: {self.target_constraints}")
            # 找出所有拥有目标标签的节点
            target_labels = set(self.target_constraints.keys())
            for n, d in self.G.nodes(data=True):
                if d.get("label") in target_labels:
                    self.candidate_seeds.append(n)
            
            if not self.candidate_seeds:
                print(f"[警告] 数据图中没有任何节点包含标签 {target_labels}！生成将非常困难。")
                self.candidate_seeds = list(self.G.nodes()) # 降级为随机
        else:
            self.candidate_seeds = list(self.G.nodes())

    def _write_fastest_query(self, qnodes: List[int], edges: List[Tuple[int, int]], outpath: str):
        node_index = {n: i for i, n in enumerate(qnodes)}
        deg = {n: 0 for n in qnodes}
        for (u, v) in edges:
            if u in deg and v in deg:
                deg[u] += 1
                deg[v] += 1

        with open(outpath, "w") as fout:
            fout.write(f"t {len(qnodes)} {len(edges)}\n")
            for n in qnodes:
                i = node_index[n]
                lab = int(self.G.nodes[n].get("label", 0))
                d = deg[n]
                fout.write(f"v {i} {lab} {d}\n")
            for (u, v) in edges:
                if u in node_index and v in node_index:
                    fout.write(f"e {node_index[u]} {node_index[v]}\n")

    def _induced_subgraph_edges(self, nodes: List[int]) -> List[Tuple[int, int]]:
        subgraph = self.G.subgraph(nodes)
        return list(subgraph.edges())

    def _check_strict_constraints(self, nodes: List[int]) -> bool:
        """
        检查生成的节点集是否严格满足标签数量要求。
        """
        if not self.target_constraints:
            return True

        current_labels = [self.G.nodes[n].get("label") for n in nodes]
        counts = Counter(current_labels)

        # 检查每个要求的标签数量是否严格匹配
        for label, req_count in self.target_constraints.items():
            if counts.get(label, 0) != req_count:
                return False
        return True

    def _get_start_node(self):
        """
        从包含目标标签的节点集合中随机选择一个起点。
        这样可以保证生成的子图至少包含一个目标标签。
        """
        if self.candidate_seeds:
            return random.choice(self.candidate_seeds)
        return random.choice(list(self.G.nodes()))

    # ------------------- 结构生成器 -------------------

    def gen_star_one(self, size: int, max_attempts: int = 2000) -> Optional[List[int]]:
        for _ in range(max_attempts):
            center = self._get_start_node() 
            neighbors = list(self.G.neighbors(center))
            
            nodes = [center]
            if len(neighbors) >= size - 1:
                nodes.extend(random.sample(neighbors, size - 1))
            else:
                # 2-hop 补充
                nodes.extend(neighbors)
                two_hop = []
                for n in neighbors:
                    two_hop.extend(self.G.neighbors(n))
                two_hop = list(set(two_hop) - set(nodes)) # 去重
                
                needed = size - len(nodes)
                if len(two_hop) >= needed:
                    nodes.extend(random.sample(two_hop, needed))
            
            # 校验连通性和标签约束
            if len(nodes) == size and nx.is_connected(self.G.subgraph(nodes)):
                if self._check_strict_constraints(nodes):
                    return nodes
        return None

    def gen_path_one(self, size: int, max_attempts: int = 2000) -> Optional[List[int]]:
        for _ in range(max_attempts):
            start_node = self._get_start_node()
            path = [start_node]
            visited = {start_node}
            
            while len(path) < size:
                curr = path[-1]
                candidates = [n for n in self.G.neighbors(curr) if n not in visited]
                if not candidates:
                    break
                path.append(random.choice(candidates))
                visited.add(path[-1])
            
            if len(path) == size and self._check_strict_constraints(path):
                return path
        return None

    def gen_cycle_one(self, size: int, max_attempts: int = 3000) -> Optional[List[int]]:
        for _ in range(max_attempts):
            start_node = self._get_start_node()
            path = [start_node]
            visited = {start_node}
            
            while len(path) < size:
                curr = path[-1]
                candidates = [n for n in self.G.neighbors(curr) if n not in visited]
                if not candidates:
                    break
                path.append(random.choice(candidates))
                visited.add(path[-1])

            # 必须闭环
            if len(path) == size and self.G.has_edge(path[-1], path[0]):
                if self._check_strict_constraints(path):
                    return path
            
            # 如果游走失败，不做复杂退化，直接依靠多次尝试和下面的 Tree 逻辑补充
        return None

    def gen_tree_one(self, size: int, max_attempts: int = 2000) -> Optional[List[int]]:
        for _ in range(max_attempts):
            start_node = self._get_start_node()
            tree_nodes = {start_node}
            frontier = deque([start_node])
            
            while len(tree_nodes) < size and frontier:
                curr = frontier.popleft()
                nbs = [n for n in self.G.neighbors(curr) if n not in tree_nodes]
                random.shuffle(nbs)
                
                expand = random.randint(1, 3)
                for n in nbs[:expand]:
                    if len(tree_nodes) < size:
                        tree_nodes.add(n)
                        frontier.append(n)
                    else:
                        break
            
            nodes = list(tree_nodes)
            if len(nodes) == size and self._check_strict_constraints(nodes):
                return nodes
        return None

    def generate_all(self, sizes: List[int], structures: List[str], N_per: int = 50, max_total_tries: int = 5000):
        gen_map = {
            "star": self.gen_star_one,
            "path": self.gen_path_one,
            "cycle": self.gen_cycle_one,
            "tree": self.gen_tree_one,
        }

        for struct in structures:
            if struct not in gen_map: continue
            func = gen_map[struct]
            
            for size in sizes:
                subdir = os.path.join(self.out_dir, f"{struct}_{size}")
                os.makedirs(subdir, exist_ok=True)
                
                count = 0
                tries = 0
                seen = set()
                
                print(f"\n[开始] {struct} size={size} (目标: {N_per} 个)...")
                
                while count < N_per and tries < max_total_tries:
                    tries += 1
                    nodes = func(size)
                    
                    if nodes:
                        key = tuple(sorted(nodes))
                        if key not in seen:
                            seen.add(key)
                            fname = os.path.join(subdir, f"query_{struct}_{size}_{count}.graph")
                            self._write_fastest_query(nodes, self._induced_subgraph_edges(nodes), fname)
                            count += 1
                            if count % 10 == 0:
                                print(f"  > 已生成 {count}/{N_per} ...")
                
                if count < N_per:
                    print(f"[警告] {struct} size={size} 未能完成目标: {count}/{N_per} (尝试次数耗尽)")
                else:
                    print(f"[完成] {struct} size={size}: 生成了 {count} 个")

# ------------------- 运行配置区 -------------------
if __name__ == "__main__":
    
    # 1. 路径设置
    DATASET_NAME = 'dataset_test2'
    DATA_GRAPH_PATH = f"/home/wangshuo/resource/datasets/parler_data/{DATASET_NAME}/data_graph/parler.graph"
    # 输出目录自动加上 label 后缀，防止混淆
    OUTPUT_BASE_DIR = "/home/wangshuo/resource/datasets/parler_data/proto_query_graphs"
    
    # ==========================
    # 2. 自定义约束配置 (修改这里)
    # ==========================
    TARGET_LABEL_ID = 3     # 你想要的标签，例如 1
    TARGET_LABEL_COUNT = 1   # 该标签必须出现的次数，例如 1 次
    
    # 3. 生成参数
    QUERY_SIZES = [10,12]
    STRUCTURES = ["star", "path", "cycle", "tree"]
    NUM_QUERIES_PER_CONFIG = 20
    
    RANDOM_SEED = 2026

    # 自动生成输出路径
    OUTPUT_DIR = os.path.join(OUTPUT_BASE_DIR, f"strict_label_{TARGET_LABEL_ID}_count_{TARGET_LABEL_COUNT}")
    
    print(f"[INFO] 正在加载图: {DATA_GRAPH_PATH}")
    try:
        data_graph = load_graph_fastest(DATA_GRAPH_PATH)
        print(f"图加载完成。")

        # 构建约束字典
        constraints = {TARGET_LABEL_ID: TARGET_LABEL_COUNT}

        generator = QueryGenerator(
            G=data_graph, 
            out_dir=OUTPUT_DIR, 
            seed=RANDOM_SEED,
            target_constraints=constraints
        )
        
        generator.generate_all(
            sizes=QUERY_SIZES, 
            structures=STRUCTURES, 
            N_per=NUM_QUERIES_PER_CONFIG,
            max_total_tries=8000
        )
        
        # --- 自动验证 ---
        print("\n--- 结果验证 ---")
        total_valid = 0
        total_files = 0
        print(f"检查目录: {OUTPUT_DIR}")
        
        for root, _, files in os.walk(OUTPUT_DIR):
            for f in files:
                if f.endswith('.graph'):
                    total_files += 1
                    qG = load_graph_fastest(os.path.join(root, f))
                    # 统计标签
                    labs = [d['label'] for n, d in qG.nodes(data=True)]
                    c = Counter(labs)
                    
                    # 检查是否满足 TARGET_LABEL_ID 出现 TARGET_LABEL_COUNT 次
                    if c.get(TARGET_LABEL_ID, 0) == TARGET_LABEL_COUNT:
                        total_valid += 1
                    else:
                        print(f"[失败] 文件 {f} 标签分布: {dict(c)} (期望 L{TARGET_LABEL_ID}={TARGET_LABEL_COUNT})")
        
        print(f"验证完毕: {total_valid}/{total_files} 个文件符合要求。")
        
    except FileNotFoundError as e:
        print(f"[错误] 文件未找到: {e}")
    except Exception as e:
        print(f"[错误] 发生异常: {e}")

#### 1.2 检验生成的查询是否满足要求：
读取/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/controlled_generation_v2/precision_structure 下的四种查询文件夹下的文件，
检查每个查询文件：
1.顶点id是否从零开始
2.顶点标签是否在原图的标签范围内0,1,2,3,4,5
3.顶点度数是否正确

In [ ]:
import os
from collections import defaultdict
from typing import Set, Tuple, List # +++ 1. 在这里导入大写的类型 +++

def verify_query_file(filepath: str, valid_labels: Set[int]) -> Tuple[bool, List[str]]:
    """
    对单个查询图文件进行三项正确性检查。

    参数:
        filepath (str): 查询图文件的完整路径。
        valid_labels (set): 一个包含所有合法标签的集合。

    返回:
        tuple[bool, list[str]]: 一个元组，第一个元素是布尔值，表示是否通过所有检查。
                               第二个元素是一个列表，包含了所有发现的错误信息。
    """
    errors = []
    
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
        
        # --- 准备工作：解析文件内容 ---
        header = lines[0].strip().split()
        num_vertices_declared = int(header[1])
        num_edges_declared = int(header[2])

        vertex_data = {} # {vid: {'label': label, 'declared_deg': deg}}
        edge_list = []
        
        for line in lines[1:]:
            parts = line.strip().split()
            if not parts:
                continue
            if parts[0] == 'v':
                if len(parts) >= 4:
                    vid, label, deg = int(parts[1]), int(parts[2]), int(parts[3])
                    vertex_data[vid] = {'label': label, 'declared_deg': deg}
                else:
                    errors.append("格式错误: 'v' 行的字段不足 (应至少有4个)")
            elif parts[0] == 'e':
                if len(parts) >= 3:
                    u, v = int(parts[1]), int(parts[2])
                    edge_list.append((u, v))
                else:
                    errors.append("格式错误: 'e' 行的字段不足 (应至少有3个)")
        
        if errors: # 如果在解析阶段就出错，提前返回
            return False, errors

        # --- 检查 1: 顶点ID是否从零开始连续 ---
        vertex_ids = sorted(vertex_data.keys())
        expected_ids = list(range(num_vertices_declared))
        if vertex_ids != expected_ids:
            errors.append(f"顶点ID检查失败: 期望ID为 {expected_ids}, 实际为 {vertex_ids}")

        # --- 检查 2: 顶点标签是否在合法范围内 ---
        for vid, data in vertex_data.items():
            if data['label'] not in valid_labels:
                errors.append(f"标签范围检查失败: 顶点 {vid} 的标签为 {data['label']}, 不在允许的范围内 {valid_labels}")
                
        # --- 检查 3: 顶点度数是否正确 ---
        actual_degrees = defaultdict(int)
        for u, v in edge_list:
            actual_degrees[u] += 1
            actual_degrees[v] += 1
            
        for vid, data in vertex_data.items():
            actual_deg = actual_degrees.get(vid, 0) # 如果是孤立点，度数为0
            declared_deg = data['declared_deg']
            if actual_deg != declared_deg:
                errors.append(f"度数检查失败: 顶点 {vid} 声明的度数为 {declared_deg}, 但根据边计算的实际度数为 {actual_deg}")

        is_valid = not bool(errors)
        return is_valid, errors

    except Exception as e:
        return False, [f"读取或解析文件时发生严重错误: {e}"]


def main_verify_all_queries():
    """
    主函数：遍历指定目录下的所有查询文件并进行验证。
    """
    # --- 配置区 ---
    queryset = 'strict_label_3_count_1'
    base_dir = f"/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/{queryset}"
    # 定义合法的标签集合
    VALID_LABELS = {0, 1, 2, 3, 4, 5}
    
    print(f"====== 开始验证目录: {base_dir} ======")
    print(f"合法标签范围: {VALID_LABELS}\n")

    total_files_checked = 0
    total_files_failed = 0
    
    # 遍历所有子目录
    for structure_dir in sorted(os.listdir(base_dir)):
        structure_path = os.path.join(base_dir, structure_dir)
        if not os.path.isdir(structure_path):
            continue
            
        print(f"--- 正在检查子目录: {structure_dir} ---")
        query_files = [f for f in os.listdir(structure_path) if f.endswith('.graph')]
        
        if not query_files:
            print("  该目录下没有找到 .graph 文件。")
            continue
            
        for query_file in sorted(query_files):
            total_files_checked += 1
            filepath = os.path.join(structure_path, query_file)
            
            is_valid, errors = verify_query_file(filepath, VALID_LABELS)
            
            if not is_valid:
                total_files_failed += 1
                print(f"  ❌ 文件检查失败: {query_file}")
                for error in errors:
                    print(f"     - {error}")
    
    print("\n" + "="*30)
    print("====== 验证总结 ======")
    print(f"总共检查文件数: {total_files_checked}")
    print(f"通过所有检查的文件数: {total_files_checked - total_files_failed}")
    print(f"检查失败的文件数: {total_files_failed}")
    
    if total_files_failed == 0:
        print("\n🎉🎉🎉 所有查询文件都满足要求！")
    else:
        print("\n⚠️⚠️⚠️ 部分查询文件存在问题，请检查上面的错误日志。")


if __name__ == '__main__':
    main_verify_all_queries()

#### 1.3 给定文件夹，统计该文件内所有查询文件中，各个标签l的节点出现的次数分布情况（=1 多少次， =2 多少次 , =3。。。。）

In [ ]:
import os
from collections import Counter
import pandas as pd # 引入 pandas 以便更好地展示结果

def analyze_label_distribution(target_dir: str):
    """
    统计指定文件夹内所有 .graph 文件中，各个标签的节点出现次数的分布情况。

    参数:
        target_dir (str): 要扫描的文件夹路径。
    """
    print(f"====== 开始分析目录: {target_dir} ======")

    # 1. 检查目录是否存在
    if not os.path.exists(target_dir):
        print(f"[错误] 目标目录不存在: {target_dir}")
        return

    # 2. 查找所有 .graph 文件
    graph_files = []
    for root, _, files in os.walk(target_dir):
        for f in files:
            if f.endswith('.graph'):
                graph_files.append(os.path.join(root, f))

    if not graph_files:
        print("在指定目录及其子目录中没有找到任何 .graph 文件。")
        return
        
    print(f"找到 {len(graph_files)} 个 .graph 文件进行分析...")

    # 3. 遍历每个文件，统计每个查询内部的标签分布
    # overall_label_distribution 将累加所有查询的标签计数
    overall_label_distribution = Counter()
    
    # query_level_counts 将记录每个查询文件中，某个标签出现了多少次
    # 例如：{ 'label_0': [1, 2, 1, ...], 'label_1': [1, 1, 1, ...], ... }
    # 列表中的每个数字代表一个查询文件中该标签的计数值
    query_level_counts = {}

    for filepath in graph_files:
        try:
            with open(filepath, 'r') as f:
                # current_query_counts 用于统计当前这一个文件内的标签计数
                current_query_counts = Counter()
                for line in f:
                    line = line.strip()
                    if line.startswith('v '):
                        parts = line.split()
                        if len(parts) >= 3:
                            try:
                                label = int(parts[2])
                                current_query_counts[label] += 1
                            except ValueError:
                                print(f"[警告] 无法解析文件 '{os.path.basename(filepath)}' 中的行: {line}")
                
                # 将当前文件的计数累加到全局计数器
                overall_label_distribution.update(current_query_counts)
                
                # 记录每个查询文件中每个标签的出现次数
                for label, count in current_query_counts.items():
                    label_key = f"label_{label}"
                    if label_key not in query_level_counts:
                        query_level_counts[label_key] = []
                    query_level_counts[label_key].append(count)

        except Exception as e:
            print(f"[错误] 处理文件 '{os.path.basename(filepath)}' 时发生异常: {e}")

    # 4. 打印全局统计结果
    print("\n" + "="*40)
    print("====== 全局标签分布统计 (所有查询总计) ======")
    if not overall_label_distribution:
        print("未能统计到任何标签。")
    else:
        # 按标签排序打印
        for label, total_count in sorted(overall_label_distribution.items()):
            print(f"  - 标签 {label}: 在所有查询中总共出现了 {total_count} 次")

    # 5. 打印每个查询中标签数量的分布情况
    print("\n" + "="*50)
    print("====== 每个查询中各标签数量的分布 (按出现次数) ======")
    
    # 将 query_level_counts 转换为一个更易于分析的结构
    # { 'label_0': Counter({1: 10, 2: 5}), 'label_1': Counter({1: 15}) }
    # 意为：有10个查询包含1个标签0的节点，5个查询包含2个标签0的节点...
    
    for label_key, counts_list in sorted(query_level_counts.items()):
        label = label_key.split('_')[1]
        # 使用 Counter 直接统计列表中每个计数值的频率
        count_distribution = Counter(counts_list)
        
        print(f"\n--- 标签 {label} 的数量分布 ---")
        if not count_distribution:
            print("  未在任何查询中找到该标签。")
        else:
            # 使用 pandas DataFrame 美化输出
            df_data = {
                '出现次数 (k)': [],
                '查询数量': [],
                '占比': []
            }
            total_queries_with_this_label = len(counts_list)
            for count_value, num_queries in sorted(count_distribution.items()):
                percentage = (num_queries / len(graph_files)) * 100
                df_data['出现次数 (k)'].append(count_value)
                df_data['查询数量'].append(f"{num_queries} 个")
                df_data['占比'].append(f"{percentage:.2f}%")
            
            df = pd.DataFrame(df_data)
            print(df.to_string(index=False))
            
    print("\n[完成] 分析结束。")


if __name__ == '__main__':
    # --- 配置区 ---
    # ✅ 在这里修改为您要分析的文件夹路径
    queryset = 'strict_label_2_count_1'
    TARGET_DIRECTORY = f"/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/{queryset}/cycle_8"
    
    # 调用主函数
    analyze_label_distribution(TARGET_DIRECTORY)

#### 2 随机生成parler_ans.txt文件 ，供fastest初步估计该查询的基数大小：

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import random

# 输入与输出路径
dataset_name = 'dataset_test3'
query_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph"
output_file = os.path.join(f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth", "parler_ans.txt")

# 获取所有查询文件
query_files = sorted(f for f in os.listdir(query_dir) if f.endswith(".graph"))

with open(output_file, "w") as f_out:
    for qf in query_files:
        # 随机生成时间(ms)和数值
        time_ms = round(random.uniform(100.0, 150000.0), 1)
        value = random.randint(10000, 800000000)
        # 写入结果
        f_out.write(f"{qf} {time_ms}ms {value}\n")

print(f"[INFO] 已生成 {len(query_files)} 条记录到 {output_file}")


[INFO] 已生成 174 条记录到 /home/wangshuo/resource/datasets/parler_data/dataset_test3/ground_truth/parler_ans.txt


#### 3 我想有规律的根据查询图生成user_custom_labels.txt文件：


##### 3.1 读取/home/wangshuo/resource/datasets/parler_data/dataset_test/query_graph 下的所有查询文件，标签2每个查询必选，
然后从标签3,4,5中随机选择一个标签加入该查询（默认概率相等且随机，可以由用户指定），最后生成的user_custom_labels.txt文件格式如下：
query_dense_1_1.graph 2 3
query_dense_1_2.graph 2 4
query_dense_1_3.graph 2 3

In [ ]:
import os
import random
from typing import List, Optional, Set, Tuple # 兼容 Python 3.8

def get_labels_from_query_file(filepath: str) -> Set[int]:
    """
    辅助函数：读取单个 .graph 文件，并返回其中所有出现过的顶点标签集合。
    """
    labels = set()
    try:
        with open(filepath, 'r') as f:
            for line in f:
                if line.startswith('v '):
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        try:
                            labels.add(int(parts[2]))
                        except ValueError:
                            continue # 忽略格式不正确的行
    except IOError:
        print(f"[警告] 无法读取文件: {filepath}")
    return labels

def generate_custom_labels_file(
    dataset_name: str, 
    must_include_label: int,
    optional_labels_pool: List[int],
    weights: Optional[List[float]] = None,
    seed: int = 42
):
    """
    根据查询图内容，为查询图动态生成 user_custom_labels.txt 文件。
    """
    print(f"====== 开始为数据集 '{dataset_name}' 生成 custom labels 文件 ======")

    # --- 1. 路径和配置 ---
    base_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
    query_dir = os.path.join(base_path, "query_graph")
    output_path = os.path.join(base_path, "data_graph", "user_custom_labels.txt")
    
    random.seed(seed)

    # --- 2. 检查与打印配置 ---
    if not os.path.exists(query_dir):
        print(f"[错误] 查询目录不存在: {query_dir}")
        return

    if weights and len(weights) != len(optional_labels_pool):
        print("[错误] 权重列表的长度必须与可选标签池的长度相同。")
        return
        
    print(f"必选标签: {must_include_label}")
    print(f"全局可选标签池: {optional_labels_pool}")
    
    # 将权重映射到标签上，方便后续动态调整
    weight_map = None
    if weights:
        weight_map = {label: weight for label, weight in zip(optional_labels_pool, weights)}
        print(f"对应权重: {weight_map}")
    else:
        print("权重: 等概率")
        
    # --- 3. 查找所有查询文件 ---
    try:
        query_files = sorted([f for f in os.listdir(query_dir) if f.endswith('.graph')])
    except FileNotFoundError:
        print(f"[错误] 无法访问查询目录: {query_dir}")
        return

    if not query_files:
        print(f"在目录 {query_dir} 中没有找到任何 .graph 文件。")
        return
    
    print(f"找到 {len(query_files)} 个查询文件进行处理...")

    # --- 4. 为每个查询文件动态生成配置行 ---
    output_lines = []
    for query_file in query_files:
        filepath = os.path.join(query_dir, query_file)
        
        # --- 核心修改：读取查询文件内容 ---
        labels_in_query = get_labels_from_query_file(filepath)
        
        # 核心标签列表总是以必选标签开始
        core_labels = [must_include_label]
        
        # --- 动态构建当前查询可用的可选池 ---
        # 找出 optional_labels_pool 中，哪些标签也存在于当前查询中
        available_optional_labels = [
            label for label in optional_labels_pool if label in labels_in_query
        ]
        
        # 从这个有效的、临时的可选池中随机选择一个
        if available_optional_labels:
            available_weights = None
            if weight_map:
                # 根据可用的标签，提取对应的权重
                available_weights = [weight_map[label] for label in available_optional_labels]

            chosen_label = random.choices(
                available_optional_labels, 
                weights=available_weights, 
                k=1
            )[0]
            core_labels.append(chosen_label)
        else:
            # 如果查询图中不包含任何可选标签，则只使用必选标签
            print(f"[信息] 查询 '{query_file}' 不包含任何可选标签 {optional_labels_pool}。只使用必选标签。")
        
        sorted_labels_str = " ".join(map(str, sorted(core_labels)))
        line = f"{query_file} {sorted_labels_str}"
        output_lines.append(line)

    # --- 5. 写入输出文件 ---
    try:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w') as f:
            for line in output_lines:
                f.write(line + "\n")
        print(f"\n✅ 成功生成配置文件，已保存到: {output_path}")
        print(f"文件包含 {len(output_lines)} 行。")

    except IOError as e:
        print(f"[错误] 无法写入文件 {output_path}: {e}")

if __name__ == '__main__':
    # --- 配置区 ---
    DATASET = 'dataset_test2'
    
    # 规则配置
    MUST_INCLUDE = 2
    OPTIONAL_POOL = [3, 4, 5] # 这是全局的可选范围
    
    # --- 权重模式 ---
    WEIGHTS = None
    # WEIGHTS = [0.5, 0.3, 0.2] # 对应 [3, 4, 5]
    
    SEED = 2024
    
    # --- 执行 ---
    generate_custom_labels_file(
        dataset_name=DATASET,
        must_include_label=MUST_INCLUDE,
        optional_labels_pool=OPTIONAL_POOL,
        weights=WEIGHTS,
        seed=SEED
    )

##### 3.2 用戶自定義選擇一個label=2，query_dense_1_1.graph 2
query_dense_1_2.graph 2
query_dense_1_3.graph 2

In [1]:
import os

def generate_custom_labels_file(
    dataset_name: str, 
    target_label: int
):
    """
    遍历查询目录下的所有 .graph 文件，并生成配置文件。
    每一行格式严格为: filename target_label
    例如: query_dense_1_1.graph 2
    """
    print(f"====== 开始为数据集 '{dataset_name}' 生成固定标签文件 (Label={target_label}) ======")

    # --- 1. 路径配置 (保持你的原始路径结构) ---
    base_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
    query_dir = os.path.join(base_path, "query_graph")
    output_path = os.path.join(base_path, "data_graph", "user_custom_labels.txt")
    
    # --- 2. 检查目录 ---
    if not os.path.exists(query_dir):
        print(f"[错误] 查询目录不存在: {query_dir}")
        return

    # --- 3. 获取所有查询文件 ---
    try:
        # 获取列表并排序，保证输出顺序整洁
        query_files = sorted([f for f in os.listdir(query_dir) if f.endswith('.graph')])
    except FileNotFoundError:
        print(f"[错误] 无法访问查询目录: {query_dir}")
        return

    if not query_files:
        print(f"在目录 {query_dir} 中没有找到任何 .graph 文件。")
        return
    
    print(f"找到 {len(query_files)} 个查询文件，准备写入标签 {target_label}...")

    # --- 4. 生成内容并写入 ---
    try:
        # 确保输出目录存在
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        with open(output_path, 'w') as f:
            for query_file in query_files:
                # 直接按照你的要求：文件名 + 空格 + 标签
                line = f"{query_file} {target_label}"
                f.write(line + "\n")
                
        print(f"\n✅ 成功生成配置文件，已保存到: {output_path}")
        print(f"文件包含 {len(query_files)} 行。")
        print("前 3 行预览:")
        for q in query_files[:3]:
            print(f"  {q} {target_label}")

    except IOError as e:
        print(f"[错误] 无法写入文件 {output_path}: {e}")

if __name__ == '__main__':
    # --- 配置区 ---
    DATASET = 'dataset_three'
    
    # 在这里自定义你想要的那个 Label
    USER_DEFINED_LABEL = 1
    
    # --- 执行 ---
    generate_custom_labels_file(
        dataset_name=DATASET,
        target_label=USER_DEFINED_LABEL
    )

====== 开始为数据集 'dataset_three' 生成固定标签文件 (Label=1) ======
找到 246 个查询文件，准备写入标签 1...

✅ 成功生成配置文件，已保存到: /home/wangshuo/resource/datasets/parler_data/dataset_three/data_graph/user_custom_labels.txt
文件包含 246 行。
前 3 行预览:
  query_cycle_4_0.graph 1
  query_cycle_4_1.graph 1
  query_cycle_4_10.graph 1


#### 4 生成推理节点配置文件：
(1)得到准确的子图匹配结果后，要计算最终的真实的答案ST_IF_Truth，下直称T_truth,需要知道每个查询图中，代表推理谓词节点的顶点编号
(2)一个查询图的格式如下，三个顶点按照id依次编号为u0,u1,u2,接下来我想读取/home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph文件夹下的所有查询文件，将标签为1的顶点所对应的标号保存到infer_node.txt，以下面查询为例要保存u1到文件中，
                                            t 3 3
                                            v 0 0 2
                                            v 1 1 2
                                            v 2 2 2
                                            e 1 2
                                            e 2 0
                                            e 1 0

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
该脚本包含两个功能：
1. extract_infer_node: 从查询文件中提取标签为1的顶点，写入 infer_node.txt。
2. generate_core_nodes_json: 根据 user_custom_labels.txt 的配置，
   为每个查询找到指定标签对应的顶点ID，并以JSON格式输出。
"""

import os
import json # 导入 json 模块

def extract_infer_node(dataset_name: str) -> str:
    """
    从指定数据集的 query_graph 文件夹下提取所有查询图中 label==1 的节点，
    并保存到 data_graph/infer_node.txt 文件中。

    参数:
        dataset_name (str): 数据集名称。

    返回:
        str: 输出文件路径。
    """
    # === 配置路径 ===
    query_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph"
    out_path = os.path.join(
        f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph",
        "infer_node.txt"
    )

    if not os.path.exists(query_dir):
        print(f"[错误] 查询目录不存在: {query_dir}")
        return ""

    # === 收集所有 .graph 文件 ===
    graph_files = [os.path.join(query_dir, f) for f in os.listdir(query_dir) if f.endswith(".graph")]
    graph_files.sort()
    print(f"[INFO] 找到 {len(graph_files)} 个查询图文件于 {query_dir}")

    # === 解析每个文件，提取 label==1 的节点编号 ===
    results = []
    for path in graph_files:
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if not line or not line.startswith("v "):
                    continue
                parts = line.split()
                if len(parts) >= 3:
                    vid = parts[1]
                    label = parts[2]
                    if label == "1":
                        results.append(f"u{vid}")
                        break  # 假设每个查询只有一个 label==1，提取后退出

    # === 写入输出文件 ===
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as fout:
        for r in results:
            fout.write(r + "\n")

    print(f"[完成] 已保存 {len(results)} 个推理节点到 {out_path}")
    return out_path


# +++ 添加下面这个新的函数 +++
def generate_core_nodes_json(dataset_name: str) -> str:
    """
    读取 user_custom_labels.txt 和查询图文件，
    生成一个JSON文件，其中包含每个查询及其指定核心标签对应的顶点ID。

    参数:
        dataset_name (str): 数据集名称。

    返回:
        str: 输出的JSON文件路径。
    """
    print("\n--- 开始生成核心节点JSON配置文件 ---")
    
    # === 配置路径 ===
    base_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
    custom_labels_path = os.path.join(base_dir, "data_graph", "user_custom_labels.txt")
    query_dir = os.path.join(base_dir, "query_graph")
    json_out_path = os.path.join(base_dir, "data_graph", "core_nodes_config.json")

    # === 1. 读取 user_custom_labels.txt 文件 ===
    if not os.path.exists(custom_labels_path):
        print(f"[错误] 配置文件不存在: {custom_labels_path}")
        return ""

    # 使用一个字典来存储配置：{ "query_filename.graph": [label1, label2, ...] }
    query_configs = {}
    with open(custom_labels_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            query_name = parts[0]
            # 将标签字符串转换为整数
            labels_to_find = {int(label) for label in parts[1:]}
            query_configs[query_name] = labels_to_find
    
    print(f"[INFO] 从 {custom_labels_path} 加载了 {len(query_configs)} 条查询配置。")

    # === 2. 遍历查询配置，解析对应的查询图文件 ===
    # 最终的JSON输出结构：{ "query_filename.graph": {"label1": [vid1, vid2], "label2": [vid3]} }
    json_output_data = {}

    for query_name, labels_to_find in query_configs.items():
        query_file_path = os.path.join(query_dir, query_name)
        
        if not os.path.exists(query_file_path):
            print(f"[警告] 配置文件中指定的查询文件不存在: {query_file_path}")
            continue
        
        # 为当前查询初始化一个结果字典
        # { 1: [], 2: [], ... }
        query_result = {label: [] for label in labels_to_find}

        with open(query_file_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line or not line.startswith("v "):
                    continue
                
                parts = line.split()
                if len(parts) >= 3:
                    try:
                        vid = int(parts[1])
                        label = int(parts[2])
                        
                        # 检查当前顶点的标签是否在我们寻找的集合中
                        if label in labels_to_find:
                            query_result[label].append(vid)
                    except ValueError:
                        print(f"[警告] 无法解析行: '{line}' in file {query_file_path}")
                        continue
        
        # 将标签(int)转换为字符串，以符合JSON key的要求
        # 同时对每个标签下的顶点ID列表进行排序，确保输出顺序稳定
        json_output_data[query_name] = {
            str(label): sorted(vids) for label, vids in query_result.items()
        }

    # === 3. 将结果写入JSON文件 ===
    os.makedirs(os.path.dirname(json_out_path), exist_ok=True)
    with open(json_out_path, "w", encoding='utf-8') as f:
        # indent=4 让JSON文件格式化，易于阅读
        json.dump(json_output_data, f, indent=4)

    print(f"[完成] 已将 {len(json_output_data)} 个查询的核心节点信息保存到 {json_out_path}")
    return json_out_path


# === 如果直接运行此脚本 ===
if __name__ == "__main__":
    # ✅ 在这里修改为你的数据集名
    dataset_name = "dataset_three" 
    
    # --- 执行第一个任务 ---
    # print("=== 任务1: 提取标签为1的推理节点 ===")
    # extract_infer_node(dataset_name)
    
    # --- 执行第二个任务 ---
    print("\n=== 任务2: 生成核心节点JSON配置 ===")
    generate_core_nodes_json(dataset_name)


=== 任务2: 生成核心节点JSON配置 ===

--- 开始生成核心节点JSON配置文件 ---
[INFO] 从 /home/wangshuo/resource/datasets/parler_data/dataset_three/data_graph/user_custom_labels.txt 加载了 246 条查询配置。
[完成] 已将 246 个查询的核心节点信息保存到 /home/wangshuo/resource/datasets/parler_data/dataset_three/data_graph/core_nodes_config.json
